# DREAMER Dataset — Full Data Exploration
**Project:** Emotion Recognition using EEG & ECG  
**Dataset:** DREAMER (23 subjects, 18 stimuli, EEG + ECG)  
**Labels:** Valence, Arousal, Dominance (scale 1–5)  
**Notebook Goal:** Load, inspect, and fully analyse the raw DREAMER.mat file  

## 0. Environment Setup & GitHub Clone
Run this cell ONLY on Google Colab

In [ ]:
import sys

# ── Colab-only setup ──────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone your GitHub repo (replace with your actual repo URL)
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/emotion-recognition.git"
    !git clone {GITHUB_REPO} /content/emotion-recognition
    %cd /content/emotion-recognition

    # Mount Google Drive (DREAMER.mat should be in Drive)
    from google.colab import drive
    drive.mount("/content/drive")

    # Point to where DREAMER.mat is stored on your Drive
    import shutil, os
    DREAMER_DRIVE_PATH = "/content/drive/MyDrive/DREAMER.mat"   # ← adjust path
    DREAMER_LOCAL_PATH = "data/raw/DREAMER.mat"
    os.makedirs("data/raw", exist_ok=True)
    shutil.copy(DREAMER_DRIVE_PATH, DREAMER_LOCAL_PATH)
    print("✅ DREAMER.mat copied to data/raw/")

    # Install mat73 if not available
    !pip install mat73 -q

else:
    DREAMER_LOCAL_PATH = "data/raw/DREAMER.mat"
    print("✅ Running locally")

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis
import warnings
import os

# DREAMER .mat loader — handles both v7 and v7.3 formats
try:
    import mat73
    MAT_LOADER = "mat73"
except ImportError:
    import scipy.io as sio
    MAT_LOADER = "scipy"

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print(f"✅ Libraries loaded | MAT loader: {MAT_LOADER}")

## 2. Load the DREAMER.mat File

In [ ]:
def load_dreamer(path: str) -> dict:
    """Load DREAMER.mat using mat73 (v7.3) or scipy (legacy)."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"DREAMER.mat not found at: {path}")
    if MAT_LOADER == "mat73":
        data = mat73.loadmat(path)
    else:
        import scipy.io as sio
        data = sio.loadmat(path, simplify_cells=True)
    return data

raw = load_dreamer(DREAMER_LOCAL_PATH)
dreamer = raw["DREAMER"]
print("✅ DREAMER.mat loaded successfully")
print(f"Top-level keys: {list(dreamer.keys())}")

## 3. Dataset-Level Metadata

In [ ]:
# ── Global metadata ───────────────────────────────────────────────
EEG_FS       = int(dreamer["EEG_SamplingRate"])   # 128 Hz
ECG_FS       = int(dreamer["ECG_SamplingRate"])   # 256 Hz
N_SUBJECTS   = int(dreamer["noOfSubjects"])       # 23
N_VIDEOS     = int(dreamer["noOfVideoSequences"]) # 18
EEG_CHANNELS = list(dreamer["EEG_Electrodes"])    # 14 channels

print("=" * 50)
print("        DREAMER — Dataset Metadata")
print("=" * 50)
print(f"  EEG Sampling Rate  : {EEG_FS} Hz")
print(f"  ECG Sampling Rate  : {ECG_FS} Hz")
print(f"  Subjects           : {N_SUBJECTS}")
print(f"  Video Stimuli      : {N_VIDEOS}")
print(f"  EEG Channels ({len(EEG_CHANNELS)})  : {EEG_CHANNELS}")
print(f"  ECG Channels       : 2 (Lead I, Lead II)")
print(f"  Labels             : Valence, Arousal, Dominance (scale 1–5)")
print("=" * 50)

## 4. Subject-Level Structure Inspection

In [ ]:
data_subjects = dreamer["Data"]   # list of 23 subject dicts

# Inspect subject 0 structure
s0 = data_subjects[0]
print("Subject-level keys:", list(s0.keys()))

# EEG & ECG sub-keys
print("\nEEG keys:", list(s0["EEG"].keys()))
print("ECG keys:", list(s0["ECG"].keys()))

# Label vectors
print(f"\nScoreValence shape  : {np.array(s0['ScoreValence']).shape}")
print(f"ScoreArousal shape  : {np.array(s0['ScoreArousal']).shape}")
print(f"ScoreDominance shape: {np.array(s0['ScoreDominance']).shape}")

## 5. Signal Shape Audit — EEG & ECG Across All Subjects

In [ ]:
records = []

for sub_idx, subject in enumerate(data_subjects):
    for vid_idx in range(N_VIDEOS):

        # EEG shapes
        eeg_baseline = np.array(subject["EEG"]["baseline"][vid_idx])
        eeg_stimuli  = np.array(subject["EEG"]["stimuli"][vid_idx])

        # ECG shapes
        ecg_baseline = np.array(subject["ECG"]["baseline"][vid_idx])
        ecg_stimuli  = np.array(subject["ECG"]["stimuli"][vid_idx])

        # Labels
        valence   = float(np.array(subject["ScoreValence"]).flatten()[vid_idx])
        arousal   = float(np.array(subject["ScoreArousal"]).flatten()[vid_idx])
        dominance = float(np.array(subject["ScoreDominance"]).flatten()[vid_idx])

        records.append({
            "subject"          : sub_idx + 1,
            "video"            : vid_idx + 1,
            "eeg_baseline_samples": eeg_baseline.shape[0],
            "eeg_stimuli_samples" : eeg_stimuli.shape[0],
            "eeg_channels"    : eeg_stimuli.shape[1] if eeg_stimuli.ndim > 1 else 1,
            "ecg_baseline_samples": ecg_baseline.shape[0],
            "ecg_stimuli_samples" : ecg_stimuli.shape[0],
            "ecg_channels"    : ecg_stimuli.shape[1] if ecg_stimuli.ndim > 1 else 1,
            "eeg_stimuli_sec" : round(eeg_stimuli.shape[0] / EEG_FS, 2),
            "ecg_stimuli_sec" : round(ecg_stimuli.shape[0] / ECG_FS, 2),
            "valence"         : valence,
            "arousal"         : arousal,
            "dominance"       : dominance,
        })

df = pd.DataFrame(records)
print(f"✅ Audit complete — Total trial records: {len(df)}")
df.head(10)

## 6. Signal Duration Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title, color in zip(
    axes,
    ["eeg_stimuli_sec", "ecg_stimuli_sec"],
    ["EEG Stimuli Duration (seconds)", "ECG Stimuli Duration (seconds)"],
    ["steelblue", "darkorange"]
):
    ax.hist(df[col], bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(df[col].mean(), color="red", linestyle="--", linewidth=1.5, label=f"Mean: {df[col].mean():.1f}s")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Duration (s)")
    ax.set_ylabel("Trial Count")
    ax.legend()

plt.suptitle("Signal Duration Distribution Across All Trials", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("outputs/results/signal_duration_distribution.png", bbox_inches="tight")
plt.show()

print("\n── EEG Stimuli Duration (seconds) ──")
print(df["eeg_stimuli_sec"].describe().round(2))
print("\n── ECG Stimuli Duration (seconds) ──")
print(df["ecg_stimuli_sec"].describe().round(2))

## 7. Label Distribution — Valence, Arousal, Dominance

In [ ]:
label_cols = ["valence", "arousal", "dominance"]
colors     = ["#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, color in zip(axes, label_cols, colors):
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index.astype(int), counts.values, color=color, edgecolor="white", width=0.6)
    ax.set_title(f"{col.capitalize()} Score Distribution", fontweight="bold")
    ax.set_xlabel("Score (1–5)")
    ax.set_ylabel("Count")
    ax.set_xticks([1, 2, 3, 4, 5])
    for i, (x, y) in enumerate(zip(counts.index, counts.values)):
        ax.text(x, y + 2, str(y), ha="center", fontsize=9)

plt.suptitle("Label Distributions — All 23 Subjects × 18 Videos", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/label_distributions.png", bbox_inches="tight")
plt.show()

print("── Label Statistics ──")
print(df[label_cols].describe().round(2))

## 8. Label Correlation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
corr = df[label_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            ax=axes[0], square=True, linewidths=0.5,
            annot_kws={"size": 12})
axes[0].set_title("Label Correlation Matrix", fontweight="bold")

# Pairplot proxy — scatter matrix
pd.plotting.scatter_matrix(
    df[label_cols], ax=None, figsize=(6, 6),
    diagonal="kde", color="steelblue", alpha=0.4
)
plt.suptitle("Label Pair Relationships", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/label_correlation.png", bbox_inches="tight")
plt.show()

print("── Correlation Matrix ──")
print(corr.round(3))

## 9. Binary Label Distribution (High / Low — threshold = 3)
Classification framing: scores > 3 → High (1), scores ≤ 3 → Low (0)

In [ ]:
THRESHOLD = 3
for col in label_cols:
    df[f"{col}_binary"] = (df[col] > THRESHOLD).astype(int)

binary_cols = [f"{c}_binary" for c in label_cols]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, binary_cols, colors):
    counts = df[col].value_counts().sort_index()
    labels_pie = [f"Low (≤{THRESHOLD})", f"High (>{THRESHOLD})"]
    ax.pie(counts.values, labels=labels_pie, autopct="%1.1f%%",
           colors=["#d9534f", "#5cb85c"], startangle=90,
           wedgeprops=dict(edgecolor="white"))
    ax.set_title(col.replace("_binary", "").capitalize(), fontweight="bold")

plt.suptitle("Binary Label Class Balance (threshold = 3)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/binary_label_balance.png", bbox_inches="tight")
plt.show()

print("── Binary Label Counts ──")
print(df[binary_cols].apply(pd.Series.value_counts))

## 10. Per-Subject Label Variability
Do individual subjects show consistent or variable emotion ratings?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, color in zip(axes, label_cols, colors):
    subject_means = df.groupby("subject")[col].mean()
    subject_stds  = df.groupby("subject")[col].std()
    ax.errorbar(
        subject_means.index, subject_means.values,
        yerr=subject_stds.values, fmt="o-", color=color,
        ecolor="gray", elinewidth=1, capsize=3
    )
    ax.axhline(3, color="black", linestyle="--", linewidth=0.8, alpha=0.5, label="Threshold=3")
    ax.set_title(f"{col.capitalize()} per Subject (mean ± std)", fontweight="bold")
    ax.set_xlabel("Subject ID")
    ax.set_ylabel("Score")
    ax.set_xticks(range(1, N_SUBJECTS + 1))
    ax.legend()

plt.suptitle("Per-Subject Label Variability", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/per_subject_label_variability.png", bbox_inches="tight")
plt.show()

## 11. Raw EEG Signal Visualisation (Subject 1 — Stimulus 1)

In [ ]:
s1   = data_subjects[0]
eeg  = np.array(s1["EEG"]["stimuli"][0])   # shape: (samples, 14)
time = np.arange(eeg.shape[0]) / EEG_FS    # seconds

PLOT_SEC  = 10
PLOT_SAMP = PLOT_SEC * EEG_FS

fig, axes = plt.subplots(7, 2, figsize=(16, 18), sharex=True)
axes = axes.flatten()

for i, ch_name in enumerate(EEG_CHANNELS):
    axes[i].plot(time[:PLOT_SAMP], eeg[:PLOT_SAMP, i],
                 color="steelblue", linewidth=0.6)
    axes[i].set_title(ch_name, fontsize=9, fontweight="bold")
    axes[i].set_ylabel("µV", fontsize=7)
    axes[i].tick_params(labelsize=7)

axes[-1].set_xlabel("Time (s)")
plt.suptitle("Raw EEG — Subject 1 | Stimulus 1 | First 10 seconds", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/raw_eeg_sample.png", bbox_inches="tight")
plt.show()

print(f"EEG shape: {eeg.shape}  →  ({eeg.shape[0]} samples × {eeg.shape[1]} channels)")
print(f"Duration : {eeg.shape[0]/EEG_FS:.2f} seconds")

## 12. Raw ECG Signal Visualisation (Subject 1 — Stimulus 1)

In [ ]:
ecg  = np.array(s1["ECG"]["stimuli"][0])   # shape: (samples, 2)
time_ecg = np.arange(ecg.shape[0]) / ECG_FS

PLOT_SEC_ECG  = 10
PLOT_SAMP_ECG = PLOT_SEC_ECG * ECG_FS

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for i, label in enumerate(["ECG Channel 1", "ECG Channel 2"]):
    axes[i].plot(time_ecg[:PLOT_SAMP_ECG], ecg[:PLOT_SAMP_ECG, i],
                 color="darkorange", linewidth=0.7)
    axes[i].set_title(label, fontweight="bold")
    axes[i].set_ylabel("mV")

axes[-1].set_xlabel("Time (s)")
plt.suptitle("Raw ECG — Subject 1 | Stimulus 1 | First 10 seconds", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/raw_ecg_sample.png", bbox_inches="tight")
plt.show()

print(f"ECG shape: {ecg.shape}  →  ({ecg.shape[0]} samples × {ecg.shape[1]} channels)")
print(f"Duration : {ecg.shape[0]/ECG_FS:.2f} seconds")

## 13. EEG Power Spectral Density (PSD) — Frequency Band Analysis
Relevant EEG bands: Delta (0.5–4 Hz), Theta (4–8 Hz), Alpha (8–13 Hz), Beta (13–30 Hz), Gamma (30–45 Hz)

In [ ]:
BANDS = {
    "Delta"  : (0.5, 4),
    "Theta"  : (4,   8),
    "Alpha"  : (8,  13),
    "Beta"   : (13, 30),
    "Gamma"  : (30, 45),
}

# PSD for one channel (AF3) across all channels for comparison
fig, axes = plt.subplots(3, 5, figsize=(20, 10), sharey=False)
axes = axes.flatten()

for ch_idx, ch_name in enumerate(EEG_CHANNELS):
    freqs, psd = scipy_signal.welch(
        eeg[:, ch_idx], fs=EEG_FS, nperseg=256
    )
    mask = freqs <= 50
    axes[ch_idx].semilogy(freqs[mask], psd[mask], color="steelblue", linewidth=0.8)
    for band, (lo, hi) in BANDS.items():
        axes[ch_idx].axvspan(lo, hi, alpha=0.15, label=band)
    axes[ch_idx].set_title(ch_name, fontweight="bold", fontsize=9)
    axes[ch_idx].set_xlabel("Hz", fontsize=7)
    axes[ch_idx].set_ylabel("PSD", fontsize=7)

# Add legend to last used axis
for band, (lo, hi) in BANDS.items():
    axes[0].axvspan(lo, hi, alpha=0.15, label=band)
axes[0].legend(fontsize=7, loc="upper right")

# Hide extra subplots
for j in range(len(EEG_CHANNELS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("EEG Power Spectral Density — All 14 Channels (Subject 1, Stimulus 1)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/eeg_psd_all_channels.png", bbox_inches="tight")
plt.show()

## 14. Band Power Summary — Per Channel

In [ ]:
def compute_band_power(signal_1d, fs, band):
    lo, hi = band
    freqs, psd = scipy_signal.welch(signal_1d, fs=fs, nperseg=256)
    idx = np.logical_and(freqs >= lo, freqs <= hi)
    return np.trapz(psd[idx], freqs[idx])

band_power_records = []
for ch_idx, ch_name in enumerate(EEG_CHANNELS):
    row = {"channel": ch_name}
    for band_name, band_range in BANDS.items():
        row[band_name] = compute_band_power(eeg[:, ch_idx], EEG_FS, band_range)
    band_power_records.append(row)

df_band = pd.DataFrame(band_power_records).set_index("channel")

# Normalised heatmap
df_band_norm = df_band.div(df_band.sum(axis=1), axis=0)

plt.figure(figsize=(10, 7))
sns.heatmap(df_band_norm, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.4, annot_kws={"size": 9})
plt.title("Relative Band Power per EEG Channel (Subject 1, Stimulus 1)", fontweight="bold")
plt.ylabel("EEG Channel")
plt.xlabel("Frequency Band")
plt.tight_layout()
plt.savefig("outputs/results/eeg_band_power_heatmap.png", bbox_inches="tight")
plt.show()

print(df_band_norm.round(3))

## 15. EEG Statistical Properties Per Channel

In [ ]:
stats_records = []
for ch_idx, ch_name in enumerate(EEG_CHANNELS):
    sig = eeg[:, ch_idx]
    stats_records.append({
        "channel" : ch_name,
        "mean"    : np.mean(sig),
        "std"     : np.std(sig),
        "min"     : np.min(sig),
        "max"     : np.max(sig),
        "skewness": skew(sig),
        "kurtosis": kurtosis(sig),
        "rms"     : np.sqrt(np.mean(sig**2)),
    })

df_stats = pd.DataFrame(stats_records).set_index("channel")
print("── EEG Statistical Properties (Subject 1, Stimulus 1) ──")
print(df_stats.round(4))

## 16. Baseline vs. Stimuli Signal Comparison (EEG)

In [ ]:
eeg_base = np.array(s1["EEG"]["baseline"][0])
eeg_stim = np.array(s1["EEG"]["stimuli"][0])

ch = 0   # AF3 for illustration

freqs_b, psd_b = scipy_signal.welch(eeg_base[:, ch], fs=EEG_FS, nperseg=256)
freqs_s, psd_s = scipy_signal.welch(eeg_stim[:, ch], fs=EEG_FS, nperseg=256)
mask = freqs_b <= 50

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time domain
t_b = np.arange(len(eeg_base)) / EEG_FS
t_s = np.arange(len(eeg_stim)) / EEG_FS
axes[0].plot(t_b[:10*EEG_FS], eeg_base[:10*EEG_FS, ch],
             color="gray", linewidth=0.6, label="Baseline")
axes[0].plot(t_s[:10*EEG_FS], eeg_stim[:10*EEG_FS, ch],
             color="steelblue", linewidth=0.6, alpha=0.85, label="Stimuli")
axes[0].set_title(f"Time Domain — Channel: {EEG_CHANNELS[ch]}", fontweight="bold")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude (µV)")
axes[0].legend()

# Frequency domain
axes[1].semilogy(freqs_b[mask], psd_b[mask], color="gray", label="Baseline")
axes[1].semilogy(freqs_s[mask], psd_s[mask], color="steelblue", label="Stimuli")
axes[1].set_title(f"PSD Comparison — Channel: {EEG_CHANNELS[ch]}", fontweight="bold")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Power (µV²/Hz)")
axes[1].legend()

plt.suptitle("EEG Baseline vs. Stimuli — Subject 1 | Video 1", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/results/eeg_baseline_vs_stimuli.png", bbox_inches="tight")
plt.show()

## 17. ECG Signal Analysis — R-Peak Detection & Heart Rate

In [ ]:
def detect_r_peaks(ecg_channel, fs):
    """Simple R-peak detection via scipy find_peaks."""
    from scipy.signal import find_peaks, butter, filtfilt
    # Bandpass filter: 0.5–40 Hz
    b, a = butter(2, [0.5 / (fs/2), 40 / (fs/2)], btype="band")
    filtered = filtfilt(b, a, ecg_channel)
    peaks, _ = find_peaks(filtered, distance=int(fs * 0.4), height=np.std(filtered) * 0.6)
    return peaks, filtered

ecg_ch1 = ecg[:, 0]
r_peaks, ecg_filtered = detect_r_peaks(ecg_ch1, ECG_FS)

# RR intervals → HR
rr_intervals = np.diff(r_peaks) / ECG_FS   # seconds
hr_bpm       = 60.0 / rr_intervals           # bpm

t_ecg_plot = np.arange(len(ecg_ch1)) / ECG_FS
PLOT_SAMP_ECG2 = 15 * ECG_FS

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ECG + R-peaks
axes[0].plot(t_ecg_plot[:PLOT_SAMP_ECG2], ecg_filtered[:PLOT_SAMP_ECG2],
             color="darkorange", linewidth=0.7, label="Filtered ECG")
valid_peaks = r_peaks[r_peaks < PLOT_SAMP_ECG2]
axes[0].scatter(valid_peaks / ECG_FS, ecg_filtered[valid_peaks],
                color="red", s=30, zorder=5, label="R-peaks")
axes[0].set_title("ECG Ch1 — Filtered Signal + R-Peak Detection (Subject 1, Stimulus 1)", fontweight="bold")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].legend()

# HR over time
hr_times = r_peaks[1:] / ECG_FS
axes[1].plot(hr_times, hr_bpm, color="#2ecc71", linewidth=1.2, marker="o", markersize=2)
axes[1].axhline(np.mean(hr_bpm), color="red", linestyle="--", linewidth=1,
                label=f"Mean HR: {np.mean(hr_bpm):.1f} bpm")
axes[1].set_title("Instantaneous Heart Rate (bpm)", fontweight="bold")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("HR (bpm)")
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/results/ecg_rpeak_hr.png", bbox_inches="tight")
plt.show()

print(f"Detected R-peaks    : {len(r_peaks)}")
print(f"Mean RR interval    : {np.mean(rr_intervals)*1000:.1f} ms")
print(f"Mean Heart Rate     : {np.mean(hr_bpm):.1f} bpm")
print(f"HR Std              : {np.std(hr_bpm):.2f} bpm")

## 18. Missing Data & Quality Check

In [ ]:
issues = []

for sub_idx, subject in enumerate(data_subjects):
    for vid_idx in range(N_VIDEOS):
        try:
            eeg_s = np.array(subject["EEG"]["stimuli"][vid_idx])
            ecg_s = np.array(subject["ECG"]["stimuli"][vid_idx])

            if np.any(np.isnan(eeg_s)) or np.any(np.isinf(eeg_s)):
                issues.append({"subject": sub_idx+1, "video": vid_idx+1,
                                "signal": "EEG", "issue": "NaN/Inf values"})
            if eeg_s.shape[1] != 14:
                issues.append({"subject": sub_idx+1, "video": vid_idx+1,
                                "signal": "EEG", "issue": f"Wrong channels: {eeg_s.shape[1]}"})
            if np.any(np.isnan(ecg_s)) or np.any(np.isinf(ecg_s)):
                issues.append({"subject": sub_idx+1, "video": vid_idx+1,
                                "signal": "ECG", "issue": "NaN/Inf values"})
            if ecg_s.shape[1] != 2:
                issues.append({"subject": sub_idx+1, "video": vid_idx+1,
                                "signal": "ECG", "issue": f"Wrong channels: {ecg_s.shape[1]}"})
        except Exception as e:
            issues.append({"subject": sub_idx+1, "video": vid_idx+1,
                           "signal": "BOTH", "issue": str(e)})

if issues:
    print(f"⚠️  {len(issues)} issues found:")
    print(pd.DataFrame(issues))
else:
    print("✅ No missing data or structural issues detected across all 23×18 trials.")

## 19. Summary Statistics — Full Dataset

In [ ]:
total_eeg_hours = df["eeg_stimuli_sec"].sum() / 3600
total_ecg_hours = df["ecg_stimuli_sec"].sum() / 3600

print("=" * 55)
print("       DREAMER — Full Dataset Summary")
print("=" * 55)
print(f"  Subjects                : {N_SUBJECTS}")
print(f"  Videos per subject      : {N_VIDEOS}")
print(f"  Total trials            : {N_SUBJECTS * N_VIDEOS}")
print(f"  EEG channels            : 14  @ {EEG_FS} Hz")
print(f"  ECG channels            : 2   @ {ECG_FS} Hz")
print(f"  Avg EEG trial duration  : {df['eeg_stimuli_sec'].mean():.1f} s ± {df['eeg_stimuli_sec'].std():.1f} s")
print(f"  Avg ECG trial duration  : {df['ecg_stimuli_sec'].mean():.1f} s ± {df['ecg_stimuli_sec'].std():.1f} s")
print(f"  Total EEG recording     : {total_eeg_hours:.2f} hours")
print(f"  Total ECG recording     : {total_ecg_hours:.2f} hours")
print(f"  Label range             : 1–5 (continuous)")
print(f"  Binary threshold        : 3 (High > 3, Low ≤ 3)")
print(f"  Valence  — mean/std     : {df['valence'].mean():.2f} / {df['valence'].std():.2f}")
print(f"  Arousal  — mean/std     : {df['arousal'].mean():.2f} / {df['arousal'].std():.2f}")
print(f"  Dominance — mean/std    : {df['dominance'].mean():.2f} / {df['dominance'].std():.2f}")
print("=" * 55)

## 20. Save Audit DataFrame & Push to GitHub

In [ ]:
import os
os.makedirs("outputs/results", exist_ok=True)

# Save the full trial audit CSV
df.to_csv("outputs/results/dreamer_trial_audit.csv", index=False)
print("✅ Saved: outputs/results/dreamer_trial_audit.csv")

# ── Push to GitHub (Colab only) ───────────────────────────────────
if IN_COLAB:
    !git config --global user.email "you@example.com"
    !git config --global user.name "Your Name"
    !git add outputs/results/ notebooks/exploration/
    !git commit -m "feat: add data exploration notebook + audit results"
    !git push
    print("✅ Pushed to GitHub")